In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ML practical/Processed_data.csv')

In [ ]:
data=data.drop(columns=['Unnamed: 0'])

In [ ]:
data.shape

(256574, 127)

In [ ]:
data.size

32584898

Import required libraries

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report ,roc_auc_score , roc_curve
from sklearn.preprocessing import label_binarize

In [ ]:
X=data.drop(columns=['Severity'])
y=data['Severity']

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)

Base models

In [ ]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=2000, class_weight='balanced'))
])

In [ ]:
base_models=[
    ('lr' ,lr_pipeline),
    ('rf' ,RandomForestClassifier(n_estimators=200,class_weight='balanced')),
    ( 'gb' ,GradientBoostingClassifier())
]

1. Meta model
2. This model will combine the predictions of the base models






In [ ]:
meta_model = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=2000))
])

STACKING CLASSIFIER

In [ ]:
stacking_model=StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    stack_method='predict_proba',
    n_jobs=-1
)

In [ ]:
stacking_model.fit(X_train,y_train)

In [ ]:
test_pred=stacking_model.predict(X_test)
train_pred=stacking_model.predict(X_train)

In [ ]:
print("Accuracy score of Test data:",accuracy_score(y_test,test_pred))
print("Accuracy score of Train data:",accuracy_score(y_train,train_pred))

CLASSIFICATION REPORT (Precision, Recall, F1 per class)

In [ ]:
print("Classification Report")
classification_report(y_test,test_pred)

In [ ]:
y_pred_prob=stacking_model.predict_proba(X_test)

Macro & Weighted Multi-Class AUC (OVR)

In [ ]:
macro_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
weighted_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
print("Macro AUC     :", macro_auc)
print("Weighted AUC  :", weighted_auc)

ROC Curve per class

In [ ]:
classes = np.unique(y_test)
y_test_bin = label_binarize(y_test, classes=classes)

for i in range(4):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_prob[:, i])
    plt.plot(fpr, tpr, label=f"Class {i+1}")

plt.plot([0,1],[0,1],'--')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
plt.show()

Confusion Matrix

In [ ]:
stack_cm=confusion_matrix(y_test,bag_pred)
sns.heatmap(stack_cm,annot=True,fmt='d',cmap='Greens')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()